In [8]:
system = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {system}")

standard_deviation = [0.229, 0.224, 0.225]
Mean = [0.485, 0.456, 0.406]
transform_image = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(Mean, standard_deviation)
])

Using device: cuda


In [9]:
# NEW, SEPARATE experiment: ResNet50 trained on bbox-cropped images
# This does NOT replace or overwrite the original whole-image ResNet50.

class Camera_Dataset_BBoxCropped(Dataset):
    def __init__(self, images_folder, path_annotations, transform=None):
        self.images_folder = images_folder
        self.transform = transform
        with open(path_annotations, "r", encoding="utf-8") as f:
            self.annotations = json.load(f)
        final_ids = sorted(set(a["category_id"] for a in self.annotations))
        self.label_map = {cat_id: slot for slot, cat_id in enumerate(final_ids)}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        annotation_data = self.annotations[index]
        img_path = os.path.join(self.images_folder, annotation_data["image_id"] + ".jpg")
        Img = Image.open(img_path).convert("RGB")

        # bbox coordinates are already in the 640x640 processed-image space
        x, y, w, h = annotation_data["bbox"]
        left = max(0, int(x))
        top = max(0, int(y))
        right = min(Img.width, int(x + w))
        bottom = min(Img.height, int(y + h))

        # Safety fallback: if the box is degenerate, use the full image instead of crashing
        if right <= left or bottom <= top:
            cropped = Img
        else:
            cropped = Img.crop((left, top, right, bottom))

        lb = self.label_map[annotation_data["category_id"]]
        if self.transform:
            cropped = self.transform(cropped)
        return cropped, lb


d_bbox = Camera_Dataset_BBoxCropped(
    r"D:\Camera_trap\data\processed_data",
    r"D:\Camera_trap\data\processed_data\prepared_annotations.json",
    transform=transform_image
)
print(f"Bbox-cropped dataset size = {len(d_bbox)}")

Bbox-cropped dataset size = 44232


In [10]:
torch.manual_seed(42)
bbox_train_size = int(len(d_bbox) * 0.8)
bbox_test_size = len(d_bbox) - bbox_train_size
dataset_train_bbox, dataset_test_bbox = random_split(d_bbox, [bbox_train_size, bbox_test_size])

dataset_train_loader_bbox = DataLoader(dataset_train_bbox, batch_size=32, shuffle=True)
dataset_test_loader_bbox = DataLoader(dataset_test_bbox, batch_size=32, shuffle=False)

print(f"Train size = {len(dataset_train_bbox)}")
print(f"Test size = {len(dataset_test_bbox)}")

Train size = 35385
Test size = 8847


In [11]:
model_bbox = models.resnet50(weights="IMAGENET1K_V1")
model_bbox.fc = nn.Linear(model_bbox.fc.in_features, 13)
model_bbox = model_bbox.to(system)

criterion_bbox = nn.CrossEntropyLoss()
optimizer_bbox = optim.Adam(model_bbox.parameters(), lr=0.0001)

num_stage = 5
for epoch in range(num_stage):
    model_bbox.train()
    running_loss = 0.0
    for images, labels in dataset_train_loader_bbox:
        images, labels = images.to(system), labels.to(system)
        optimizer_bbox.zero_grad()
        outputs = model_bbox(images)
        loss = criterion_bbox(outputs, labels)
        loss.backward()
        optimizer_bbox.step()
        running_loss += loss.item()
    average_loss = running_loss / len(dataset_train_loader_bbox)
    print(f"Epoch [{epoch+1}/{num_stage}]  Loss: {average_loss:.4f}")

torch.save(model_bbox.state_dict(), r"D:\Camera_trap\models\resnet50_bbox_cropped_trained.pth")
print("Bbox-cropped model saved (separate file, original untouched)")

Epoch [1/5]  Loss: 0.4105
Epoch [2/5]  Loss: 0.1787
Epoch [3/5]  Loss: 0.1210
Epoch [4/5]  Loss: 0.0900
Epoch [5/5]  Loss: 0.0769
Bbox-cropped model saved (separate file, original untouched)


In [12]:
model_bbox.eval()
bbox_predictions = []
bbox_labels_true = []

with torch.no_grad():
    for images, labels in dataset_test_loader_bbox:
        images, labels = images.to(system), labels.to(system)
        outputs = model_bbox(images)
        _, predicted = torch.max(outputs, 1)
        bbox_predictions.extend(predicted.cpu().numpy())
        bbox_labels_true.extend(labels.cpu().numpy())

accuracy_bbox = sum(p == t for p, t in zip(bbox_predictions, bbox_labels_true)) / len(bbox_labels_true) * 100
print(f"Bbox-cropped accuracy: {accuracy_bbox:.2f}%")

from sklearn.metrics import classification_report
print(classification_report(bbox_labels_true, bbox_predictions, labels=list(range(13))))

Bbox-cropped accuracy: 94.62%
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      2309
           1       0.96      0.97      0.97      1434
           2       0.89      0.95      0.92       482
           3       0.91      0.94      0.93       869
           4       0.91      0.93      0.92       149
           5       0.99      0.88      0.94       486
           6       0.95      0.93      0.94       932
           7       0.95      0.95      0.95       968
           8       0.90      0.89      0.90       282
           9       0.91      0.92      0.91       810
          10       0.82      0.87      0.85        38
          11       0.00      0.00      0.00         1
          12       0.92      0.84      0.88        87

    accuracy                           0.95      8847
   macro avg       0.85      0.85      0.85      8847
weighted avg       0.95      0.95      0.95      8847



D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
from sklearn.metrics import classification_report
print(classification_report(bbox_labels_true, bbox_predictions, labels=list(range(13))))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      2309
           1       0.96      0.97      0.97      1434
           2       0.89      0.95      0.92       482
           3       0.91      0.94      0.93       869
           4       0.91      0.93      0.92       149
           5       0.99      0.88      0.94       486
           6       0.95      0.93      0.94       932
           7       0.95      0.95      0.95       968
           8       0.90      0.89      0.90       282
           9       0.91      0.92      0.91       810
          10       0.82      0.87      0.85        38
          11       0.00      0.00      0.00         1
          12       0.92      0.84      0.88        87

    accuracy                           0.95      8847
   macro avg       0.85      0.85      0.85      8847
weighted avg       0.95      0.95      0.95      8847



D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\anaconda3\envs\cameratrap\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
